<a href="https://colab.research.google.com/github/VJN1089/CNS/blob/main/CNSPROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q gradio pyotp

In [16]:
!pip install -q gradio

In [2]:
import sqlite3
import secrets
import hashlib
import hmac
import getpass
DATABASE = "/content/users.db"
ITERATIONS = 600_000
def get_db():
    return sqlite3.connect(DATABASE)
with get_db() as db:
    db.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE NOT NULL,
            salt BLOB NOT NULL,
            password_hash BLOB NOT NULL
        )
    """)
print("Database created successfully!")
print("Secure Login System is ready.")

Database created successfully!
Secure Login System is ready.


In [3]:
def hash_password(password, salt):
    return hashlib.pbkdf2_hmac(
        "sha256",
        password.encode("utf-8"),
        salt,
        ITERATIONS
    )
print("Password hashing function is ready.")

Password hashing function is ready.


In [4]:
def register_user(username, password):
    username = username.strip()
    if not username or not password:
        print("Username and password are required.")
        return False
    if len(username) > 80 or len(password) > 1024:
        print("Input is too long.")
        return False
    salt = secrets.token_bytes(16)
    password_hash = hash_password(password, salt)
    try:
        with get_db() as db:
            db.execute(
                """INSERT INTO users
                   (username, salt, password_hash)
                   VALUES (?, ?, ?)""",
                (username, salt, password_hash)
            )
        print("Registration successful!")
        print("Salt and password hash stored.")
        return True
    except sqlite3.IntegrityError:
        print("Username already exists.")
        return False
print("Registration function is ready.")

Registration function is ready.


In [5]:
def login_user(username, password):
    with get_db() as db:
        user = db.execute(
            """SELECT salt, password_hash
               FROM users WHERE username = ?""",
            (username.strip(),)
        ).fetchone()
    if user is None:
        print("Login failed: invalid username or password.")
        return False
    stored_salt, stored_hash = user
    entered_hash = hash_password(password, stored_salt)
    if hmac.compare_digest(entered_hash, stored_hash):
        print("Login successful! Welcome,", username)
        return True
    print("Login failed: invalid username or password.")
    return False
print("Login function is ready.")

Login function is ready.


In [10]:
username = input("Enter a new username: ").strip()
password = getpass.getpass("Enter a password: ")
register_user(username, password)

Enter a new username: VJN
Enter a password: ··········
Registration successful!
Salt and password hash stored.


True

In [11]:
login_name = input("Enter your username: ").strip()
login_password = getpass.getpass("Enter your password: ")
login_user(login_name, login_password)

Enter your username: VJN
Enter your password: ··········
Login successful! Welcome, VJN


True

In [12]:
with get_db() as db:
    rows = db.execute(
        "SELECT username, salt, password_hash FROM users"
    ).fetchall()
print("Stored database records:\n")
for username, salt, password_hash in rows:
    print("Username:", username)
    print("Salt (hex):", salt.hex())
    print("Password hash (hex):", password_hash.hex())
    print("Plaintext password: NOT STORED")
    print("-" * 50)

Stored database records:

Username: Jagadeesh
Salt (hex): e5bfda82818e50d48cd400eedb0a9204
Password hash (hex): 7ce8f634812d59a59862f9ec8c23034e1468284eb0ac75099c71cfb91af147a8
Plaintext password: NOT STORED
--------------------------------------------------
Username: VJN
Salt (hex): 7b80a88d75217370e329af59318158ac
Password hash (hex): 52724d26f975ebfe48278b2edf2e41666e51428d26ed283ee6d2c0a1f2a7a3fb
Plaintext password: NOT STORED
--------------------------------------------------


In [13]:
demo_password = "Example@123"
salt1 = secrets.token_bytes(16)
salt2 = secrets.token_bytes(16)
hash1 = hash_password(demo_password, salt1)
hash2 = hash_password(demo_password, salt2)
print("Same password:", demo_password)
print("Hash with salt 1:", hash1.hex())
print("Hash with salt 2:", hash2.hex())
print("Hashes are different:", hash1 != hash2)

Same password: Example@123
Hash with salt 1: a7e0e97f1cd72b0f8db2e6309b67fd3d3a4b77e807f9edf54dc211f6e82109d7
Hash with salt 2: ef084d9ca5a27ae3eba2f608494e74bcc340957c978729402fffe4bd15db74ee
Hashes are different: True


In [18]:

import sqlite3
import secrets
import hashlib
import hmac
import random
import gradio as gr

DATABASE = "/content/users.db"
ITERATIONS = 600_000

def get_db():
    return sqlite3.connect(DATABASE)

# Keep your existing users database
with get_db() as db:
    db.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE NOT NULL,
            salt BLOB NOT NULL,
            password_hash BLOB NOT NULL
        )
    """)

def hash_password(password, salt):
    return hashlib.pbkdf2_hmac(
        "sha256", password.encode("utf-8"),
        salt, ITERATIONS
    )

def register(username, password):
    username = username.strip()

    if not username or not password:
        return "Enter username and password."

    if len(username) > 80 or len(password) > 1024:
        return "Input is too long."

    salt = secrets.token_bytes(16)
    password_hash = hash_password(password, salt)

    try:
        with get_db() as db:
            db.execute("""
                INSERT INTO users (username, salt, password_hash)
                VALUES (?, ?, ?)
            """, (username, salt, password_hash))

        return "Registration successful! You can now log in."

    except sqlite3.IntegrityError:
        return "Username already exists."

def new_captcha():
    a = random.randint(1, 9)
    b = random.randint(1, 9)
    return f"What is {a} + {b}?", str(a + b)

def login(username, password, captcha_input, expected_answer):
    username = (username or "").strip()

    # Check CAPTCHA first
    if not (captcha_input or "").strip().isdigit():
        return "Please enter the correct CAPTCHA answer."

    if not hmac.compare_digest(
        (captcha_input or "").strip(),
        expected_answer or ""
    ):
        return "Incorrect CAPTCHA. Refresh and try again."

    # Verify password
    with get_db() as db:
        user = db.execute("""
            SELECT salt, password_hash
            FROM users
            WHERE username = ?
        """, (username,)).fetchone()

    if user is None:
        return "Login failed: invalid username or password."

    salt, stored_hash = user
    entered_hash = hash_password(password or "", salt)

    if not hmac.compare_digest(entered_hash, stored_hash):
        return "Login failed: invalid username or password."

    return f"Login successful! Welcome, {username}."

def refresh_captcha():
    question, answer = new_captcha()
    return question, answer, ""

initial_question, initial_answer = new_captcha()

with gr.Blocks(title="Secure Login System") as app:
    gr.Markdown("""
    # 🔐 Secure Login System
    ### Password Hashing + CAPTCHA Verification
    """)

    with gr.Tab("Register"):
        reg_username = gr.Textbox(label="Create username")
        reg_password = gr.Textbox(
            label="Create password", type="password"
        )
        reg_button = gr.Button("Register")
        reg_result = gr.Textbox(label="Registration result")

        reg_button.click(
            register,
            inputs=[reg_username, reg_password],
            outputs=reg_result
        )

    with gr.Tab("Login"):
        username = gr.Textbox(label="Username")
        password = gr.Textbox(
            label="Password", type="password"
        )

        question = gr.Textbox(
            label="CAPTCHA Challenge",
            value=initial_question,
            interactive=False
        )

        # Keep the correct answer server-side
        answer_state = gr.State(initial_answer)

        captcha_input = gr.Textbox(
            label="Enter CAPTCHA answer",
            placeholder="Enter the answer"
        )

        with gr.Row():
            login_button = gr.Button("Login securely")
            refresh_button = gr.Button("New CAPTCHA")

        login_result = gr.Textbox(
            label="Login result", interactive=False
        )

        login_button.click(
            login,
            inputs=[
                username, password, captcha_input, answer_state
            ],
            outputs=login_result
        )

        refresh_button.click(
            refresh_captcha,
            outputs=[question, answer_state, captcha_input]
        )

app.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6d68bec0b3105146bc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
